# Running models and querying results

Compile an age-stratified SIR, run it with `CompiledModel.run`, inspect the
save plan with `describe` before solving, then query the `Result` by date,
age, calendar month, and a rolling window. Close with a jitted loss that ends
in `at_times`.


In [ ]:
from datetime import date

import jax
import jax.numpy as jnp
import numpy as np

from summer4 import (
    Compartments,
    Epoch,
    FlowModel,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
pmap = PropertyMap.from_property(state).stratify(age)

model = FlowModel(pmap)
model.add_flow(TransitionFlow("infection", state["S"], state["I"], 0.25))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.1))
cm = model.compile()

y0 = np.zeros(pmap.size)
y0[pmap.select(state["S"])] = 999.0
y0[pmap.select(state["I"])] = 1.0
y0 = PropertyData.wrap(pmap, y0)

epoch = Epoch(date(2020, 1, 1))
plan = SavePlan(requests={"compartments": SaveRequest(Compartments())})
print(cm.describe(plan, y0=y0, n_saves=61))


## Run and plot prevalence

In [ ]:
res = cm.run({}, y0, t0=0.0, steps=60, dt=1.0, save=plan, epoch=epoch)
assert "compartments" in res
assert res.solver is not None and res.solver.num_steps == 60

prevalence = res["compartments"].select(state["I"]).total()
frame = prevalence.to_frame()
assert frame.height == 61
_ = prevalence.plot(legend=False)


## Select by date and aggregate by age

In [ ]:
jan = (
    res["compartments"]
    .select(state["I"])
    .sum_over(age)
    .between(date(2020, 1, 1), date(2020, 1, 31))
)
assert np.asarray(jan.times.values).size == 31
at_mid = res["compartments"].select(state["I"]).total().at(date(2020, 1, 15))
assert float(np.asarray(at_mid.values).reshape(-1)[0]) > 0


## Calendar month resample and 7-day rolling mean

In [ ]:
monthly = res["compartments"].select(state["I"]).total().resample("ME", how="sum")
assert monthly.times.values.shape[0] == 3  # Jan, Feb, and 1 Mar day
rolled = res["compartments"].select(state["I"]).total().rolling(7, how="mean", min_periods=1)
assert np.asarray(rolled.values).shape == np.asarray(prevalence.values).shape


## JIT loss ending in `at_times`

In [ ]:
def loss(scale: jax.Array) -> jax.Array:
    # Scale the whole initial state (unique indices for scatter grads).
    y = PropertyData(pmap, y0.data * scale)
    out = cm.run({}, y, t0=0.0, steps=30, dt=1.0, save=plan)
    pred = out["compartments"].select(state["I"]).total().at_times(np.array([30.0]))
    return jnp.sum(jnp.asarray(pred.values) ** 2)

jitted = jax.jit(loss)
val = float(jitted(jnp.asarray(1.0)))
grad = float(jax.grad(loss)(jnp.asarray(1.0)))
assert np.isfinite(val) and np.isfinite(grad)
print(f"loss={val:.4g}, grad={grad:.4g}")
